In [ ]:
# Import package
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import igraph
import scvelo as scv
import loompy as lmp
import anndata as ad
import harmonypy as hm
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os

import celltypist
from celltypist import models
from celltypist.models import Model

In [ ]:
Liver_sn_raw=sc.read("/home/jaehyunchoi/Liver_bio_project/50_Public_GEO_analysis/analysis/Data/Liver_bio_nucseq_raw.h5ad")
Liver_sn_raw

In [ ]:
Liver_sn_raw.obs['celltype'].value_counts()

In [ ]:
adata_hepa = Liver_sn_raw[(Liver_sn_raw.obs['celltype'] == 'Hepatocyte')].copy()
adata_hepa

In [ ]:
adata_hepa.raw = adata_hepa
sc.pp.normalize_total(adata_hepa, target_sum=1e4)
sc.pp.log1p(adata_hepa)

In [ ]:
# 변수 유전자 설정 
sc.pp.highly_variable_genes(adata_hepa, flavor='seurat', n_top_genes=2000)
# Scale data
sc.pp.scale(adata_hepa, max_value=10)
# Run PCA
sc.tl.pca(adata_hepa, svd_solver='arpack')
print(adata_hepa.obsm['X_pca'][:5, :5])
sc.pl.pca(adata_hepa, color='sample')

import matplotlib.pyplot as plt
var_ratio = adata_hepa.uns['pca']['variance_ratio']
plt.plot(range(1, len(var_ratio) + 1), np.cumsum(var_ratio), marker='o')
plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('PCA Cumulative Variance')
plt.grid(True)
plt.show()

In [ ]:
sc.pl.umap(
    adata_hepa,
    color='Disease_status',
    frameon=False,
    title='Cell type prediction',
    size=10
)

In [ ]:
sc.pp.neighbors(adata_hepa, n_pcs=30, use_rep='X_pca')
sc.tl.tsne(adata_hepa, use_rep='X_pca', n_pcs=30)

sc.pl.tsne(
    adata_hepa,
    color='Disease_status',
    frameon=False,
    title='Cell type prediction',
    size=10
)

In [ ]:
adata_hepa.write("/home/jaehyunchoi/Liver_bio_project/50_Public_GEO_analysis/analysis/Data/Liver_bio_hepatocyte.h5ad")

In [ ]:
sc.tl.leiden(adata_hepa, resolution=0.1, key_added="leiden")
sc.pl.umap(adata_hepa, color=['leiden'])

In [ ]:
import os
import tempfile

import anndata
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scvi
import seaborn as sns
import torch

In [ ]:
adata_hepa=sc.read("/home/jaehyunchoi/Liver_bio_project/50_Public_GEO_analysis/analysis/Data/Liver_bio_hepatocyte.h5ad")
adata_hepa

In [ ]:
sc.pl.umap(
    adata_hepa,
    color='Disease_status',
    frameon=False,
    title='Cell type prediction',
    size=10
)

In [ ]:
sc.pl.umap(
    adata_hepa,
    color='GLUL',
    frameon=False,
    title='Cell type prediction',
    size=10
)

In [ ]:
sc.pl.violin(
    adata_hepa,
    keys=['SLC38A2', 'SLC38A5'],
    groupby='Disease_status',
    multi_panel=True,      # 유전자마다 subplot (세로로 쌓임)
    stripplot=False,       # 점 생략
    rotation=0
)

In [ ]:
Liver_hepa = adata_hepa[adata_hepa.obs['Disease_status'] != 'MASLD'].copy()
# 순서 정의
stage_order = ["Healthy", "MASL", "MASH", "Cirrhosis", "end stage"]
Liver_hepa.obs['stage'] = Liver_hepa.obs['Disease_status']
Liver_hepa.obs['stage'] = pd.Categorical(Liver_hepa.obs['stage'], categories=stage_order, ordered=True)
Liver_hepa.obs['stage_numeric'] = Liver_hepa.obs['stage'].cat.codes

In [ ]:
sc.pl.violin(
    Liver_hepa,
    keys=['SLC38A2', 'SLC38A5'],
    groupby='stage',
    multi_panel=True,      # 유전자마다 subplot (세로로 쌓임)
    stripplot=False,       # 점 생략
    rotation=0
)

In [ ]:
sc.pl.violin(
    Liver_hepa,
    keys=['SLC7A5'],
    groupby='stage',
    multi_panel=True,      # 유전자마다 subplot (세로로 쌓임)
    stripplot=False,       # 점 생략
    rotation=0
)

In [ ]:
# t-sne 
sc.tl.tsne(Liver_hepa, use_rep='X_pca', n_pcs=30)
# umap
sc.pp.neighbors(Liver_hepa, n_pcs=30, use_rep='X_pca')
sc.tl.umap(Liver_hepa)
sc.pl.umap(Liver_hepa, color=['journal'])

In [ ]:
Liver_hepa.write("/home/jaehyunchoi/Liver_bio_project/50_Public_GEO_analysis/analysis/Data/Liver_bio_hepatocyte_umap.h5ad")

In [ ]:
Liver_hepa=sc.read("/home/jaehyunchoi/Liver_bio_project/50_Public_GEO_analysis/analysis/Data/Liver_bio_hepatocyte_umap.h5ad")
Liver_hepa

In [ ]:
sc.pl.umap(Liver_hepa, color=['journal'])

In [ ]:
Liver_hepa.obs

In [ ]:
sc.pl.embedding(Liver_hepa, basis='X_scANVI', color='journal', dimensions=[(0, 1)])

In [ ]:
import matplotlib.pyplot as plt
var_ratio = Liver_hepa.uns['']['variance_ratio']
plt.plot(range(1, len(var_ratio) + 1), np.cumsum(var_ratio), marker='o')
plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('PCA Cumulative Variance')
plt.grid(True)
plt.show()

In [ ]:
# Harmony로 batch effect 제거
harmony_out = hm.run_harmony(Liver_hepa.obsm['X_umap'],Liver_hepa.obs, 'journal')

# Harmony 결과를 저장
Liver_hepa.obsm['X_pca_harmony'] = harmony_out.Z_corr.T

# UMAP을 사용한 시각화
sc.pp.neighbors(Liver_hepa, use_rep='X_pca_harmony')  # Harmony 결과를 사용
sc.tl.umap(Liver_hepa)
sc.pl.umap(Liver_hepa, color=['journal'])

In [ ]:
sc.tl.leiden(Liver_hepa, resolution=0.05, key_added="hepa_sub")
sc.pl.umap(Liver_hepa, color=['hepa_sub'])

In [ ]:
# Harmony로 batch effect 제거
harmony_out = hm.run_harmony(Liver_hepa.obsm['X_pca'],Liver_hepa.obs, 'sample')

# Harmony 결과를 저장
Liver_hepa.obsm['X_pca_harmony'] = harmony_out.Z_corr.T

# UMAP을 사용한 시각화
sc.pp.neighbors(Liver_hepa, use_rep='X_pca_harmony')  # Harmony 결과를 사용
sc.tl.umap(Liver_hepa)
sc.pl.umap(Liver_hepa, color=['journal'])

In [ ]:
sc.tl.leiden(Liver_hepa, resolution=0.05, key_added="hepa_sub")
sc.pl.umap(Liver_hepa, color=['hepa_sub'])

In [ ]:
Liver_hepa.obs['hepa_sub'].value_counts()

In [ ]:
sc.pl.umap(Liver_hepa, color=['GLUL'])

In [ ]:
Markers = {
    'Zone1': ['CPS1', 'HAL', 'ARG1', 'SLC38A2', 'SLC1A5'],
    'Zone3': ['GLUL', 'CYP2E1', 'CYP1A2', 'CD36', 'SLC7A5', 'SLC16A10'],
    'Bile':  ['ABCB11', 'ABCC2', 'ALB', 'KRT19']
}

In [ ]:
Zonal_Markers = {
    'periportal': ["HAL","MFSD2A","IGFBP2","PCK1","ASS1","SDS"],
    'pericentral': ["CYP3A4","SLC1A2","SLCO1B3","AXIN2","LGR5","GLUL"]}

In [ ]:
Zone_1_gene = ['CPS1', 'HAL', 'ARG1', 'SLC38A2', 'SLC1A5']
Zone_3_gene = ['GLUL', 'CYP2E1', 'CYP1A2', 'CD36', 'SLC7A5', 'SLC16A10']
Bile_gene = ['ABCB11', 'ABCC2', 'ALB', 'KRT19']

In [ ]:
sc.pl.dotplot(
    Liver_hepa,
    Markers,
    groupby="stage",
    standard_scale= None
)

In [ ]:
sc.pl.dotplot(
    Liver_hepa,
    Markers,
    groupby="hepa_sub",
    standard_scale= "var"
)

In [ ]:
sc.pl.dotplot(
    Liver_hepa,
    Zonal_Markers,
    groupby="stage",
    standard_scale= None
)

In [ ]:
genes = ['SLC7A5', 'SLC6A19', 'SLC1A5', 'SLC7A8', 'SLC1A4', 'SLC16A10', 'SLC43A1', 'SLC3A2', 'SLC7A6']
n_genes = len(genes)
fig, axs = plt.subplots(n_genes, 1, figsize=(6, 1.8 * n_genes))  # 세로 크기를 줄임

# axs가 하나일 경우 (genes가 1개) 대비
if n_genes == 1:
    axs = [axs]

for i, gene in enumerate(genes):
    sc.pl.violin(
        Liver_hepa,
        keys=gene,
        groupby='stage',
        ax=axs[i],
        show=False,
        stripplot=False,
        jitter=False
    )
    axs[i].set_title(gene)

plt.tight_layout()
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/IPF_data/analysis/figures/CD4+9cluster_violinplot.pdf", bbox_inches='tight', dpi=300)
plt.show()

# non scanvi umap trial

In [ ]:
# t-sne 
sc.tl.tsne(adata_hepa, use_rep='X_pca', n_pcs=30)
# umap
sc.pp.neighbors(adata_hepa, n_pcs=30, use_rep='X_pca')
sc.tl.umap(adata_hepa)
sc.pl.umap(adata_hepa, color=['sample'])

In [ ]:
sc.pl.umap(adata_hepa, color=['journal'])

In [ ]:
adata_hepa.obs

In [ ]:
# Harmony로 batch effect 제거
harmony_out = hm.run_harmony(adata_hepa.obsm['X_pca'],adata_hepa.obs, 'sample')

# Harmony 결과를 저장
adata_hepa.obsm['X_pca_harmony'] = harmony_out.Z_corr.T

# UMAP을 사용한 시각화
sc.pp.neighbors(adata_hepa, use_rep='X_pca_harmony')  # Harmony 결과를 사용
sc.tl.umap(adata_hepa)
sc.pl.umap(adata_hepa, color=['journal'])

In [ ]:
sc.tl.leiden(adata_hepa, resolution=0.1, key_added="leiden")
sc.pl.umap(adata_hepa, color=['leiden'])

In [ ]:
sc.pl.umap(adata_hepa, color=['leiden'])